In [1]:
#yfinance, pandas, matplotlib 다운받기
!pip install yfinance pandas matplotlib

In [4]:
#yfinance랑 pandas를 약어로 불러오기
import yfinance as yf
import pandas as pd

In [5]:
#yf에 있는 수많은 데이터중에서 내가 택한 주식 종목 불러오기
tickers = ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'AMZN']
data = yf.download(tickers, start='2025-01-01', end='2026-03-18')

#특히 종가에 집중하기 
data_price = data['Close']
data_price.head()

[*********************100%***********************]  5 of 5 completed


Ticker,AAPL,AMZN,GOOGL,MSFT,NVDA
Date,,,,,
2025-01-02,242.525162,220.220001,188.558258,414.568573,138.264694
2025-01-03,242.037811,224.190002,190.907394,419.292877,144.422684
2025-01-06,243.668915,227.610001,195.964020,423.749786,149.381042
2025-01-07,240.894073,222.110001,194.590393,418.322296,140.094086
2025-01-08,241.381424,222.130005,193.057465,420.491272,140.064117


In [6]:
#주식의 섹터, 시가총액의 정보도 궁금한데, .info는 개별종목마다 호출하니까 for 문 사용해서 리스트로 만들기

#일단 빈 리스트에 이름 붙여주기 
meta_data = []

for ticker in tickers :
    stock_info = yf.Ticker(ticker).info

#리스트안에 하나씩 넣어주기
    meta_data.append({
        'Ticker':ticker,
        'Sector':stock_info.get('sector','N/A'),
        'MarketCap':stock_info.get('marketCap',0)
    })
 #보기 좋게 dataframe으로 바꾸기
info=pd.DataFrame(meta_data)
info

,Ticker,Sector,MarketCap
0,AAPL,Technology,3644938780672
1,MSFT,Technology,2838202155008
2,NVDA,Technology,4197473583104
3,GOOGL,Communication Services,3641197199360
4,AMZN,Consumer Cyclical,2204630515712


In [7]:
#marketcap의 단위가 너무 크길래 조단위의 단위로 바꾸어주기
info['MarketCap_BN']=info['MarketCap'] / 1e9

#이제 불러온 정보들 다 보기 
info[['Ticker', 'Sector', 'MarketCap_BN']] 

,Ticker,Sector,MarketCap_BN
0,AAPL,Technology,3644.938781
1,MSFT,Technology,2838.202155
2,NVDA,Technology,4197.473583
3,GOOGL,Communication Services,3641.197199
4,AMZN,Consumer Cyclical,2204.630516


In [8]:
#결측치 확인하기
print(data_price.isnull().sum())


Ticker
AAPL     0
AMZN     0
GOOGL    0
MSFT     0
NVDA     0
dtype: int64


# Financial Data Cleaning Module
## Step 2 : Data Cleaning &nbsp;|&nbsp; Step 2.5 : Feature Engineering

---

**목적** : Step 1에서 수집한 원시 주가 데이터를 통계 분석에 적합한 형태로 정제하고,  
섹터·시가총액 메타데이터와 결합하여 분석 가능한 마스터 테이블을 완성한다.

| 단계 | 내용 | 핵심 개념 |
|------|------|-----------|
| Step 1 *(완료)* | yfinance로 OHLCV + 메타데이터 수집 | Time-series 원재료 확보 |
| **Step 2** | 결측치 처리, 타입 강제, 무결성 검사 | Data Integrity |
| **Step 2.5** | 병합, 파생변수 생성, 이상치 탐지 | Feature Engineering |

---
## ▶ Step 1 Recap — 수집한 데이터 불러오기

> Step 1에서 작성한 코드를 그대로 실행해서 `data_price`와 `info` 객체를 이 노트북으로 가져온다.  
> *(Step 1 코드가 별도 파일에 있다면 해당 셀을 여기에 붙여넣기)*

In [1]:
# ── Step 1 코드 (그대로 재실행) ────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')   # yfinance FutureWarning 숨기기

import yfinance as yf
import pandas as pd
import numpy as np

# 종목 리스트 & 기간
tickers = ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'AMZN']

# 원시 OHLCV 다운로드
data = yf.download(tickers, start='2025-01-01', end='2026-03-18', auto_adjust=True, progress=False)

# 종가(Close) 추출
data_price = data['Close']

# 메타데이터 (Sector, MarketCap)
meta_data = []
for ticker in tickers:
    stock_info = yf.Ticker(ticker).info
    meta_data.append({
        'Ticker'    : ticker,
        'Sector'    : stock_info.get('sector', 'N/A'),
        'MarketCap' : stock_info.get('marketCap', 0)
    })

info = pd.DataFrame(meta_data)
info['MarketCap_BN'] = info['MarketCap'] / 1e9

print('=== data_price shape :', data_price.shape)
print('=== info shape       :', info.shape)
data_price.head(3)

=== data_price shape : (301, 5)
=== info shape       : (5, 4)


Ticker,AAPL,AMZN,GOOGL,MSFT,NVDA
Date,,,,,
2025-01-02,242.525146,220.220001,188.558273,414.568604,138.264709
2025-01-03,242.037811,224.190002,190.907410,419.292908,144.422684
2025-01-06,243.668900,227.610001,195.964020,423.749756,149.381042


---
# STEP 2 — Data Cleaning
## 🎯 목표 : 데이터 무결성(Data Integrity) 확보

원시 데이터에는 세 가지 문제가 잠재할 수 있다.

1. **결측치(NaN)** — 주말·공휴일에 시장이 닫히면서 생기는 빈 칸
2. **잘못된 데이터 타입** — CSV 재로드 시 숫자가 문자열로 읽히는 경우
3. **재무적으로 불가능한 값** — `High < Low`, `Close ≤ 0` 등

이 세 가지를 순서대로 해결한다.

### 2-A. 결측치 현황 파악 — Before Snapshot

클리닝을 시작하기 **전에** NaN이 얼마나 있는지 기록해둔다.  
나중에 'Before → After'를 비교해야 클리닝이 실제로 효과가 있었음을 **증명**할 수 있다.  
*(재현 가능한 연구(Reproducible Research)의 기본 원칙)*

In [2]:
# ── 2-A : 결측치 Before Snapshot ──────────────────────────────────────────

missing_before = data_price.isnull().sum() #결측치 개수 합계
total_cells    = data_price.size #총 데이터 사이즈

audit_before = pd.DataFrame({
    'Missing_Count' : missing_before,
    'Missing_Pct'   : (missing_before / len(data_price) * 100).round(2)
}) #표로 결측 비율 보기, 비율 소수점 둘째자리까지 반올림

print('▶ 결측치 현황 (클리닝 전)')
print(audit_before)
print(f'\n전체 셀 수 : {total_cells:,}  |  NaN 합계 : {missing_before.sum()}')

▶ 결측치 현황 (클리닝 전)
        Missing_Count  Missing_Pct
Ticker                            
AAPL                0          0.0
AMZN                0          0.0
GOOGL               0          0.0
MSFT                0          0.0
NVDA                0          0.0

전체 셀 수 : 1,505  |  NaN 합계 : 0


### 2-B. 결측치 처리 — Forward Fill (ffill)

**왜 `ffill()`을 쓰는가?**

주식 시장은 주말·공휴일에 거래가 없다. 이 날의 *경제적 사실*은  
"마지막으로 거래된 가격이 여전히 유효하다"는 것이다.

- ✅ `ffill()` → 직전 거래일 가격을 그대로 전진 복사 (경제적으로 올바름)
- ❌ 선형 보간(interpolation) → 주말 동안 가격이 부드럽게 변했다고 *가정* (사실과 다름)
- ❌ 0으로 채우기 → 통계 왜곡 (평균·분산 모두 망가짐)

> 추가로 `bfill()`을 뒤에 적용하는 이유:  
> 시리즈의 **맨 앞**에 NaN이 있을 때 (예: 상장일이 우리 시작일보다 늦은 종목)  
> ffill만으로는 채울 수 없기 때문에 bfill로 뒷받침한다.

In [3]:
# ── 2-B : ffill → bfill 순서로 결측치 처리 ────────────────────────────────

# data_price는 Wide 형태 (행=날짜, 열=종목) → 열(종목)별로 독립 처리됨
data_price_clean = data_price.ffill().bfill() #결측치 처리함

# After Snapshot
missing_after = data_price_clean.isnull().sum() #이제 후의 결측치 비율이랑 전이랑 비교

comparison = pd.DataFrame({
    'Before' : missing_before,
    'After'  : missing_after,
    'Resolved': missing_before - missing_after
})

print('▶ 결측치 Before → After 비교')
print(comparison)
print(f"\n✓ 잔여 NaN : {missing_after.sum()} (0이면 완전 해결)")

▶ 결측치 Before → After 비교
        Before  After  Resolved
Ticker                         
AAPL         0      0         0
AMZN         0      0         0
GOOGL        0      0         0
MSFT         0      0         0
NVDA         0      0         0

✓ 잔여 NaN : 0 (0이면 완전 해결)


### 2-C. 데이터 타입(Dtype) 확인 및 강제 변환

pandas는 데이터를 읽을 때 의도치 않게 숫자 컬럼을 `object(문자열)` 타입으로 인식할 수 있다.  
이 경우 `pct_change()`, `mean()` 같은 수치 연산이 **조용히 실패**하거나 엉뚱한 값을 낸다.  
명시적 타입 강제로 계약을 확정한다.

In [4]:
# ── 2-C : Dtype 확인 ───────────────────────────────────────────────────────

print('▶ 현재 dtype:')
print(data_price_clean.dtypes)

# float64가 아닌 열이 있으면 강제 변환
for col in data_price_clean.columns:
    if data_price_clean[col].dtype != 'float64':
        data_price_clean[col] = pd.to_numeric(data_price_clean[col], errors='coerce')
        print(f'  → {col} : 강제 변환 완료')

print('\n▶ 변환 후 dtype:')
print(data_price_clean.dtypes)

▶ 현재 dtype:
Ticker
AAPL     float64
AMZN     float64
GOOGL    float64
MSFT     float64
NVDA     float64
dtype: object

▶ 변환 후 dtype:
Ticker
AAPL     float64
AMZN     float64
GOOGL    float64
MSFT     float64
NVDA     float64
dtype: object


### 2-D. 재무 데이터 무결성 검사 (Integrity Check)

금융 데이터에는 *수학적으로 반드시 성립해야 하는 제약 조건*이 있다.  
이를 어기는 행이 있다면 데이터 벤더 오류, 스플릿 처리 실수, API 버그일 가능성이 높다.

| 검사 항목 | 조건 | 위반 시 의미 |
|-----------|------|--------------|
| 가격 양수 | `Close > 0` | 데이터 오염 또는 상장폐지 |
| 이상 급락 | 일일 수익률 `< -50%` | 스플릿 미보정 가능성 |
| 이상 급등 | 일일 수익률 `> +100%` | 역스플릿 미보정 가능성 |

In [5]:
# ── 2-D : 무결성 검사 ─────────────────────────────────────────────────────

issues_found = 0 

# 검사 1 : 종가 > 0
non_positive = (data_price_clean <= 0).sum()
print('▶ [검사 1] Close > 0')
if non_positive.sum() == 0:
    print('   ✓ PASS — 모든 종가가 양수')
else:
    print(f'   ✗ FAIL — {non_positive[non_positive > 0]}')
    issues_found += non_positive.sum()

# 검사 2, 3 : 일일 수익률 범위 (-50% ~ +100%)
daily_ret_check = data_price_clean.pct_change() #ret=return(수익률) 줄임말. #그러므로 결측치 처리한 데이터의 퍼센테이지 변화를 불러오는 것
extreme_drop    = (daily_ret_check < -0.50).sum()
extreme_rise    = (daily_ret_check >  1.00).sum()

print('\n▶ [검사 2] 일일 수익률 < -50% (이상 급락)')
if extreme_drop.sum() == 0:
    print('   ✓ PASS')
else:
    print(f'   ✗ FAIL — {extreme_drop[extreme_drop > 0]}')
    issues_found += extreme_drop.sum()

print('\n▶ [검사 3] 일일 수익률 > +100% (이상 급등)')
if extreme_rise.sum() == 0:
    print('   ✓ PASS')
else:
    print(f'   ✗ FAIL — {extreme_rise[extreme_rise > 0]}')
    issues_found += extreme_rise.sum()

print(f'\n총 이슈 수 : {issues_found}  →  {"이상 없음 ✓" if issues_found == 0 else "⚠ 검토 필요"}')

▶ [검사 1] Close > 0
   ✓ PASS — 모든 종가가 양수

▶ [검사 2] 일일 수익률 < -50% (이상 급락)
   ✓ PASS

▶ [검사 3] 일일 수익률 > +100% (이상 급등)
   ✓ PASS

총 이슈 수 : 0  →  이상 없음 ✓


### ✅ Step 2 완료 — 중간 저장

`data_price_clean` : 결측치 없음, dtype 정확, 무결성 통과한 종가 DataFrame  
CSV로 저장해두면 Step 2.5를 별도 세션에서 시작할 때도 재다운로드 없이 바로 이어갈 수 있다.

In [7]:
# ── Step 2 결과 저장 ───────────────────────────────────────────────────────
data_price_clean.to_csv('step2_cleaned.csv') #csv로 저장. data_price_clean 표를 내 컴퓨터 어딘가에 step2_cleaned.csv 파일로 저장
print('✓ step2_cleaned.csv 저장 완료')
print(f'  Shape : {data_price_clean.shape}') #표가 몇행 몇열인지(행=날짜, 열=종목수)
print(f'  기간  : {data_price_clean.index[0].date()} ~ {data_price_clean.index[-1].date()}')
#총 기간이 언제부터 언제인지
#data_price_clean.index → 날짜들의 목록
# [2025-01-02, 2025-01-03, 2025-01-06, ...]

#data_price_clean.index[0] → 목록에서 첫 번째 날짜
# 2025-01-02 00:00:00  ← 시간까지 붙어있음

#data_price_clean.index[0].date() → 날짜만 깔끔하게 추출
# 2025-01-02

#data_price_clean.index[-1]→ [-1] 은 파이썬에서 "맨 마지막" 이라는 뜻
# 2026-03-18 00:00:00

#data_price_clean.index[-1].date() → 2026-03-18

data_price_clean.tail(3) 

✓ step2_cleaned.csv 저장 완료
  Shape : (301, 5)
  기간  : 2025-01-02 ~ 2026-03-17


Ticker,AAPL,AMZN,GOOGL,MSFT,NVDA
Date,,,,,
2026-03-13,250.119995,207.669998,302.279999,395.549988,180.250000
2026-03-16,252.820007,211.740005,305.559998,399.950012,183.220001
2026-03-17,254.229996,215.199997,310.920013,399.410004,181.929993


---
# STEP 2.5 — Feature Engineering & Structuring
## 🎯 목표 : 분석 가능한 마스터 테이블 완성

현재 `data_price_clean`은 **Wide 형태** (행=날짜, 열=종목) 이다.  
통계 분석 라이브러리 대부분은 **Long 형태** (행=관측값 1개, 열=변수) 를 기대한다.

```
[Wide]                         [Long]
Date     | AAPL | MSFT  →   Date       | Ticker | Close | Sector | ...
2025-01-02| 243  | 415       2025-01-02 | AAPL   | 243   | Tech   | ...
2025-01-03| 247  | 418       2025-01-02 | MSFT   | 415   | Tech   | ...
```

wide 형태면 종목별로 나뉨. 볼 수 있는 값이 하나(가격 등)
근데 long 형태로 변환하면 날짜별로 쭉 나타나니 종목별로 볼 수 있는게 여러개가 됨
그리고 대부분의 통계라이브러리에서 이거 씀

변환 후 섹터·시가총액을 붙이고, 파생변수를 추가하여 마스터 테이블을 완성한다.

### 2.5-A. Wide → Long 변환 (Tidy Data)

`stack()`을 사용해 열(종목)을 행으로 접는다.  
이 구조를 **Tidy Data** 라고 하며, `groupby()`, `seaborn`, `statsmodels` 모두 이 형태를 표준으로 받는다.

In [9]:
# ── 2.5-A : Wide → Long 변환 ──────────────────────────────────────────────

# stack() : 열 레이블(종목명)을 행 인덱스로 접음
df_long = (
    data_price_clean
    .stack()                          # (Date, Ticker) MultiIndex
    .reset_index()                    # 인덱스(행) → 컬럼(열)으로 풀기. 날짜랑 종목이 행 이름에 있었음.
    .rename(columns={
        'level_1' : 'Ticker',         # stack 후 열 이름이 이상하게 나오길래 이름 다음과 같이 고정
        0         : 'Close'
    })
)

# Ticker 컬럼 이름이 level_1이랑 다를 수 있어서 방어 처리
if 'Ticker' not in df_long.columns:
    col_candidates = [c for c in df_long.columns if c not in ['Date', 'Close']] # Date도 아니고 Close도 아닌 컬럼을 찾아서
    df_long.rename(columns={col_candidates[0]: 'Ticker'}, inplace=True) #그 컬럼 이름을 'Ticker'로 바꿔줘
 
df_long['Date'] = pd.to_datetime(df_long['Date'])# 날짜가 문자열 "2025-01-02" 로 저장돼 있을 수 있어서, 진짜 날짜 타입으로 바꿔주는 것
df_long = df_long.sort_values(['Ticker', 'Date']).reset_index(drop=True) # Ticker 기준으로 먼저 정렬, 그 안에서 날짜 순으로 정렬. AAPL 날짜순 → AMZN 날짜순 → GOOGL 날짜순 ...
#.reset_index(drop=True): 정렬하고 나면 행 번호가 뒤죽박죽이 되는데 0, 1, 2, 3 ... 으로 깔끔하게 다시 매기기. drop=True : 원래 행 번호는 버리기

print(f'▶ Wide {data_price_clean.shape}  →  Long {df_long.shape}')
print(f'   컬럼 : {list(df_long.columns)}')
df_long.head(6)

▶ Wide (301, 5)  →  Long (1505, 3)
   컬럼 : ['Date', 'Ticker', 'Close']


,Date,Ticker,Close
0,2025-01-02,AAPL,242.525146
1,2025-01-03,AAPL,242.037811
2,2025-01-06,AAPL,243.668900
3,2025-01-07,AAPL,240.894073
4,2025-01-08,AAPL,241.381424
5,2025-01-10,AAPL,235.563187


### 2.5-B. 메타데이터 병합 (Merge)

Step 1에서 만든 `info` DataFrame (Sector, MarketCap_BN)을 가격 데이터에 붙인다.

가격 표랑 회사정보 표를 종목명 기준으로 합치는 것!

**왜 Left Join인가?**  
- 가격 데이터의 모든 행을 보존한다. (Inner Join은 메타가 없는 종목 행을 통째로 잃는다)  
- 메타가 없으면 NaN으로 표시 → 문제가 *보이게* 된다 (숨기는 것보다 낫다)
- 왼쪽 표(df_long. Date, Ticker, Close 있는 표)의 행을 기준으로 오른쪽 표의 정보를 오른쪽에 붙이는 것
- 왼쪽 행은 다 살아남으니!

In [11]:
# ── 2.5-B : 메타데이터 Left Join ──────────────────────────────────────────

# info에서 필요한 열만 선택(info에는 원래 MarketCap(원본) 컬럼도 있음. 그치만 이미 MarketCap_BN으로 변환했으니 원본은 필요 없음)  
meta_slim = info[['Ticker', 'Sector', 'MarketCap_BN']].copy()

df_master = df_long.merge(  # merge: 합치기
    meta_slim,
    on       = 'Ticker',    # Ticker 컬럼을 기준으로 합치기
    how      = 'left',      # df_long이 왼쪽으로. 그러므로 가격 행 전부 보존
    validate = 'm:1'        # m:1은 many to one 이란 뜻. df_long의 AAPL 행이 300개면, 300개 행 전부에 같은 AAPL 메타정보 1개를 붙이는 것 = 정상으로. Ticker당 메타 1행임을 assert (중복 방지 안전장치)
)

# 병합 후 행 수 동일 확인
assert len(df_master) == len(df_long), '병합 후 행 수가 달라짐 — 중복 확인 필요' #assert = "이게 맞지 않으면 즉시 에러 내줘" 라는 뜻

print(f'▶ 병합 완료  shape : {df_master.shape}')
print(f'   컬럼 목록 : {list(df_master.columns)}')
df_master.head(6)

▶ 병합 완료  shape : (1505, 5)
   컬럼 목록 : ['Date', 'Ticker', 'Close', 'Sector', 'MarketCap_BN']


,Date,Ticker,Close,Sector,MarketCap_BN
0,2025-01-02,AAPL,242.525146,Technology,3644.938781
1,2025-01-03,AAPL,242.037811,Technology,3644.938781
2,2025-01-06,AAPL,243.668900,Technology,3644.938781
3,2025-01-07,AAPL,240.894073,Technology,3644.938781
4,2025-01-08,AAPL,241.381424,Technology,3644.938781
5,2025-01-10,AAPL,235.563187,Technology,3644.938781


### 2.5-C. 파생변수 생성 (Derived Features)

| 파생변수 | 공식 | 통계적 의미 |
|----------|------|-------------|
| `Daily_Return` | $(P_t - P_{t-1}) / P_{t-1}$ | 산술 수익률 — 직관적, 설명 쉬움 |
| `Log_Return` | $\ln(P_t / P_{t-1})$ | 로그 수익률 — 시간합산 가능, 정규분포 근사, 회귀모델 표준 입력 |
| `Rolling_Vol_21d` | $\sigma(\text{log\_r}, 21) \times \sqrt{252}$ | 21거래일 롤링. 21일 간 얼마나 들쑥날쑥했는지(=변동성. 위험성). 이것을 연간 변동성으로 환산해서 연환산 변동성 (리스크 지표) |
| `Cumulative_Return` | $(P_t / P_0) - 1$ | 시작점 대비 누적 수익률 (차트에 활용) |

> **핵심** : 모든 계산은 `groupby('Ticker')`로 종목별 독립 처리.  
> 그렇지 않으면 AAPL의 마지막 날과 AMZN의 첫날이 이어붙어서 수익률이 오염된다.

In [12]:
# ── 2.5-C : 파생변수 생성 ─────────────────────────────────────────────────

df_master = df_master.sort_values(['Ticker', 'Date']).copy() #Ticker 순 -> 날짜 순으로 정렬한거 copy

# (i) 일일 산술 수익률
df_master['Daily_Return'] = (
    df_master.groupby('Ticker')['Close'] #groupby('Ticker') 쓰면 AAPL끼리만/ AMZN끼리만 계산. 절대 섞이지 않음
    .pct_change()           # (P_t - P_{t-1}) / P_{t-1}
)

# (ii) 일일 로그 수익률
df_master['Log_Return'] = (
    df_master.groupby('Ticker')['Close']
    .transform(lambda x: np.log(x / x.shift(1))) #shift(1) = 한 칸 아래로 밀기 = "어제 값 가져오기". 그러므로 오늘 종가 / 어제 종가 → 로그 씌우기
) 

# (iii) 21거래일 롤링 연환산 변동성
df_master['Rolling_Vol_21d'] = (
    df_master.groupby('Ticker')['Log_Return']
    .transform(
        lambda x: x.rolling(window=21, min_periods=10).std() * np.sqrt(252) #min_periods=10: 데이터 시작 시점엔 21일치가 없음 최소 10일치만 있어도 계산하게 해주는 옵션  # np.sqrt(252)는 일별 변동성 → 연간 변동성으로 환산할때 1년 거래일이 252일이라서 √252 를 곱함 # .std() 는 그 21일 동안 수익률이 얼마나 들쑥날쑥했는지 = 변동성(위험도)   
    )
)

# (iv) 누적 수익률 (첫 관측일 기준)
df_master['Cumulative_Return'] = (
    df_master.groupby('Ticker')['Close']
    .transform(lambda x: (x / x.iloc[0]) - 1)  #.iloc[0] = 첫 번째 행의 값 = 1월 2일 시작 가격
)

print('▶ 파생변수 추가 완료')
print(f'   최종 컬럼 : {list(df_master.columns)}')
df_master[['Date','Ticker','Close','Daily_Return','Log_Return',
           'Rolling_Vol_21d','Cumulative_Return','Sector','MarketCap_BN']].tail(8)

▶ 파생변수 추가 완료
   최종 컬럼 : ['Date', 'Ticker', 'Close', 'Sector', 'MarketCap_BN', 'Daily_Return', 'Log_Return', 'Rolling_Vol_21d', 'Cumulative_Return']


,Date,Ticker,Close,Daily_Return,Log_Return,Rolling_Vol_21d,Cumulative_Return,Sector,MarketCap_BN
1497,2026-03-06,NVDA,177.810379,-0.030108,-0.030571,0.441094,0.286014,Technology,4197.473583
1498,2026-03-09,NVDA,182.640106,0.027162,0.026800,0.446550,0.320945,Technology,4197.473583
1499,2026-03-10,NVDA,184.760010,0.011607,0.011540,0.361870,0.336277,Technology,4197.473583
1500,2026-03-11,NVDA,186.029999,0.006874,0.006850,0.351585,0.345463,Technology,4197.473583
1501,2026-03-12,NVDA,183.139999,-0.015535,-0.015657,0.354508,0.324561,Technology,4197.473583
1502,2026-03-13,NVDA,180.250000,-0.015780,-0.015906,0.356214,0.303659,Technology,4197.473583
1503,2026-03-16,NVDA,183.220001,0.016477,0.016343,0.358135,0.325139,Technology,4197.473583
1504,2026-03-17,NVDA,181.929993,-0.007041,-0.007066,0.350470,0.315809,Technology,4197.473583


### 2.5-D. 이상치 탐지 (Outlier Detection) — IQR Fence

**왜 z-score 대신 IQR을 쓰는가?**

주식 일일 수익률은 **두꺼운 꼬리(fat-tail / leptokurtic)** 분포를 따른다.  
z-score는 평균과 표준편차를 사용하는데, 이 두 값 자체가 극단값에 크게 민감하다.  
→ 이상치가 이상치 탐지 기준을 오염시키는 악순환.

$$IQR = Q_3 - Q_1$$ IQR은 극단값에 **강건(robust)** 하고 정규분포 가정이 없다.  

$$\text{Lower fence} = Q_1 - 3 \times IQR, \quad \text{Upper fence} = Q_3 + 3 \times IQR$$

이상치는 **삭제하지 않고** `Is_Outlier` 플래그로만 표시한다.  
삭제 여부는 분석 목적에 따라 분석자가 결정해야 한다.

In [16]:
# ── 2.5-D : IQR 기반 이상치 탐지 ─────────────────────────────────────────

def flag_outliers_iqr(group: pd.Series, k: float = 3.0) -> pd.Series:

    #group: pd.Series   # 입력값 = 한 종목의 수익률 시리즈
    #k: float = 3.0     # IQR에 곱할 숫자 (기본값 3.0)
    #-> pd.Series       # 출력값 = True/False 시리즈
    
    """종목별 IQR 펜스를 계산해 이상치 여부(bool)를 반환한다."""
    q1, q3 = group.quantile(0.25), group.quantile(0.75)
    iqr    = q3 - q1 
    return (group < q1 - k * iqr) | (group > q3 + k * iqr)  #  | 는 OR. 둘 중 하나라도 True면 True
    #그러므로 pd.Series라고 하는 것은 return 중 하나라도 만족하면 True. 아니면 False 반환

df_master['Is_Outlier'] = (
    df_master.groupby('Ticker')['Daily_Return']
    .transform(flag_outliers_iqr)  # 위에서 만든 함수에 넣기
    .fillna(False)   # 첫 행은 수익률 NaN. NaN 대신 False(이상치 아님)로 채워주기 → 이상치 아님으로 처리
)

# 결과 요약
outlier_summary = (
    df_master.groupby('Ticker')['Is_Outlier']
    .agg(Flagged='sum', Total='count') #agg는 Aggregate(집계)의 약자. 말 그대로 흩어져 있는 수많은 데이터를 통계량(평균, 합계, 분산 등)으로 요약할 때 사용
    .assign(Pct=lambda x: (x['Flagged'] / x['Total'] * 100).round(2)) #assign은 기존 데이터를 바탕으로 새로운 변수(열)를 추가할 때 사용
)

print('▶ 이상치 탐지 결과 (종목별)')
print(outlier_summary)
print(f'\n전체 플래그 수 : {df_master["Is_Outlier"].sum()}')

▶ 이상치 탐지 결과 (종목별)
        Flagged  Total   Pct
Ticker                      
AAPL          8    301  2.66
AMZN          3    301  1.00
GOOGL         4    301  1.33
MSFT          4    301  1.33
NVDA          2    301  0.66

전체 플래그 수 : 21


---
## ✅ 최종 검증 리포트 (Final Validation Report)

print('=' * 62)          = 구분선 출력

1. Schema       → 표의 구조 확인
2. Missing      → 빈칸 없는지 확인
3. 기술통계     → 숫자들 평균/최대/최소 확인
4. 커버리지     → 종목별 데이터 기간 확인
5. 섹터 분포    → 섹터 잘 붙었는지 확인

In [14]:
# ── 최종 검증 리포트 ───────────────────────────────────────────────────────

print('=' * 62)
print('  FINAL VALIDATION REPORT')
print('=' * 62)

# 1. Schema
print('\n── 1. SCHEMA')
df_master.info(show_counts=True) #show_counts=True 는 각 열에 값이 몇 개 있는지

# 2. Missing values
print('\n── 2. MISSING VALUES (마스터 테이블)')
missing_final = df_master.isnull().sum()
price_cols    = ['Close', 'Daily_Return', 'Log_Return']
if missing_final[price_cols].sum() == 0:
    print('  ✓ 가격·수익률 컬럼 : NaN 없음')
else:
    print(missing_final[missing_final > 0])

# 3. 기술통계
print('\n── 3. 기술통계 (핵심 컬럼)')
stat_cols = ['Close', 'Daily_Return', 'Log_Return', 'Rolling_Vol_21d', 'MarketCap_BN']
print(df_master[stat_cols].describe().round(5))

# 4. 종목별 커버리지
print('\n── 4. 종목별 커버리지')
coverage = (
    df_master.groupby('Ticker')['Date']
    .agg(Rows='count', Start='min', End='max') #데이터 기간 확인 #Rows='count' :행 수 세기
)
print(coverage)

# 5. 섹터 분포
print('\n── 5. 섹터 분포')
print(df_master.groupby('Ticker')['Sector'].first().to_string()) #.first():첫 번째 값만 가져오기. 같은 종목은 Sector가 다 똑같으니까 1개만 보면 됨 #.string(): 텍스트로 출력

print('\n' + '=' * 62)
print('  ✓ 마스터 테이블 완성 — 분석 준비 완료')
print('=' * 62)

  FINAL VALIDATION REPORT

── 1. SCHEMA
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1505 entries, 0 to 1504
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Date               1505 non-null   datetime64[ns]
 1   Ticker             1505 non-null   object        
 2   Close              1505 non-null   float64       
 3   Sector             1505 non-null   object        
 4   MarketCap_BN       1505 non-null   float64       
 5   Daily_Return       1500 non-null   float64       
 6   Log_Return         1500 non-null   float64       
 7   Rolling_Vol_21d    1455 non-null   float64       
 8   Cumulative_Return  1505 non-null   float64       
 9   Is_Outlier         1505 non-null   bool          
dtypes: bool(1), datetime64[ns](1), float64(6), object(2)
memory usage: 107.4+ KB

── 2. MISSING VALUES (마스터 테이블)
Daily_Return        5
Log_Return          5
Rolling_Vol_21d    50
dtype: int64


## 💾 최종 저장

In [15]:
# ── 마스터 테이블 저장 ─────────────────────────────────────────────────────
save_cols = [
    'Date', 'Ticker', 'Close',
    'Daily_Return', 'Log_Return', 'Rolling_Vol_21d', 'Cumulative_Return',
    'Sector', 'MarketCap_BN', 'Is_Outlier'
]

df_master[save_cols].to_csv('step2_5_master.csv', index=False)

print('✓ step2_5_master.csv 저장 완료')
print(f'  Shape  : {df_master.shape}')
print(f'  컬럼   : {save_cols}')
print()
print('▶ 샘플 (종목별 마지막 3행):')
print(df_master.groupby('Ticker')[save_cols].tail(3).to_string(index=False))

✓ step2_5_master.csv 저장 완료
  Shape  : (1505, 10)
  컬럼   : ['Date', 'Ticker', 'Close', 'Daily_Return', 'Log_Return', 'Rolling_Vol_21d', 'Cumulative_Return', 'Sector', 'MarketCap_BN', 'Is_Outlier']

▶ 샘플 (종목별 마지막 3행):
      Date Ticker      Close  Daily_Return  Log_Return  Rolling_Vol_21d  Cumulative_Return                 Sector  MarketCap_BN  Is_Outlier
2026-03-13   AAPL 250.119995     -0.022052   -0.022299         0.297249           0.031316             Technology   3644.938781       False
2026-03-16   AAPL 252.820007      0.010795    0.010737         0.248129           0.042449             Technology   3644.938781       False
2026-03-17   AAPL 254.229996      0.005577    0.005562         0.236627           0.048262             Technology   3644.938781       False
2026-03-13   AMZN 207.669998     -0.008877   -0.008917         0.261844          -0.056988      Consumer Cyclical   2204.630516       False
2026-03-16   AMZN 211.740005      0.019598    0.019409         0.255326          -0.

---
## 🗺️ 다음 단계 (Step 3 — Visualization)

`step2_5_master.csv` 를 입력으로, 아래 분석을 수행할 예정:

| 시각화 | 활용 데이터 | 목적 |
|--------|------------|------|
| 섹터별 일일 수익률 박스플롯 | `Daily_Return` × `Sector` | 섹터 간 분포 비교 |
| 시가총액 대비 변동성 산점도 | `MarketCap_BN` × `Rolling_Vol_21d` | 기업 규모-리스크 관계 |
| 누적 수익률 라인 차트 | `Cumulative_Return` × `Date` | 종목별 성과 비교 |
| 수익률 상관관계 히트맵 | `Daily_Return` 피벗 | 포트폴리오 다각화 분석 |

> GitHub README에 이 노트북 링크 + 위 로드맵 표를 올리면  
> 교수님이 *"이 학생은 분석 흐름을 설계할 줄 안다"* 는 인상을 받게 됩니다.